In [1]:
import pandas as pd
import tensorflow as tf
import os

2026-04-28 02:10:32.646653: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 02:10:32.657313: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777342232.671489      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777342232.674982      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 02:10:32.688930: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
#Hyperparameters
output_dir = "./"
test_dir = "../input/plant-pathology-2021-fgvc8/test_images/"
model_dir = "../input/conve01/eff7-e17/epoch-17"
resnet50_weights = "../input/resnet50weights/last_epoch-20"
efficientB7 = "../input/efficientb7/effb7"
resnet50 = "../input/resnet50/Model-Resnet"

image_dims = (300,300,3)
data_set = pd.read_csv("../input/plant-pathology-2021-fgvc8/train.csv")
df_labels = data_set['labels']
one_hot = df_labels.str.get_dummies(sep=" ")
dataset_labels = one_hot.columns.to_list()
tf.__version__

'2.18.0'

In [3]:
from tensorflow.keras import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, BatchNormalization, Dropout, GlobalMaxPool2D, Conv2D, InputLayer

class MultiLabel(Model):
    def __init__(self):
        super().__init__()
        self.model_backbone = tf.keras.models.load_model(filepath=resnet50)
        self.model = Sequential()
        self.model.add(InputLayer(input_shape=image_dims))
        self.model.add(self.model_backbone)
        self.model.add(Conv2D(filters=1024, kernel_size=(1, 1), padding='same'))
        self.model.add(BatchNormalization(momentum=0.7))
        self.model.add(Dropout(0.2))
        self.model.add(Conv2D(filters=256, kernel_size=(1, 1), padding='same'))
        self.model.add(BatchNormalization(momentum=0.7))
        self.model.add(Dropout(0.1))
        self.model.add(Conv2D(filters=128, kernel_size=(1, 1), padding='same'))
        self.model.add(GlobalMaxPool2D())
        self.model.add(Dense(units=6, activation='sigmoid'))


    def call(self, predict_input):
        predict_output = self.model(predict_input)
        return predict_output

    def create_model(self):
        return self.model


In [4]:
if __name__ == '__main__':

    # Initialization of model and data creator class.
    model = MultiLabel()
    model.build(input_shape=[None, image_dims[0], image_dims[1], image_dims[2]])
    model.load_weights(resnet50_weights)
    images_path_list = sorted(list(os.listdir(test_dir)))
    def test_on_sub (index):
        input_img = tf.io.read_file(test_dir+images_path_list[index])
        image = tf.io.decode_image(contents=input_img, channels=3, dtype=tf.dtypes.float32)
        tensor_image = tf.image.resize(image, [image_dims[0], image_dims[1]])
        name_jpg = images_path_list[index].split(os.path.sep)[-1]
        return name_jpg, tf.expand_dims(tensor_image, axis=0)
    
    values = []
    for i in range(len(images_path_list)):
        name, images = test_on_sub(index=i)
        test_values = model.call(images)
        print(test_values)
        index_values = [i for i, v in enumerate(test_values[0]) if v > 0.7]
        classes = dataset_labels
        classes_img = ""
        for i in index_values:
            classes_img = str(classes[i])+" "+classes_img
        values.append([name, classes_img])
    csv_pd = pd.DataFrame(values, columns=['image', 'labels'], index=None)
    csv_pd.to_csv(output_dir + 'submission.csv',index=False)

ValueError: File format not supported: filepath=../input/resnet50/Model-Resnet. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(../input/resnet50/Model-Resnet, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).